# IMQCAM DMS API Endpoints

This notebook calls each REST endpoint on the IMQCAM DMS directly using the Girder client, showing what parameters each accepts and what the raw response looks like.

Data reaches the DMS in two shapes, and they are served by two different sets of endpoints:

- **Forms and form entries** — the curated, structured records. A *form* is a schema; an *entry* is one filled-in instance of it, a JSON payload with typed fields. This is where process parameters, test results and measured values live. Read with `/form` and `/entry`.
- **Raw and derived data files** — the actual measurement files. Raw data is what an instrument produced (`ebsd_raw`, `fractography_raw`, `ct_tomography`); derived data is what a processing pipeline made from it (`ebsd_derived`). These are Girder items tagged with a `data_type` and an `igsn`, not form fields. Read with the `/imqcam` endpoints.

The two are joined by IGSN: an entry describes a specimen, and the data files carry the same specimen's identifier. Neither shape contains the other — an analysis that needs both has to query both.

| Endpoint | Description |
|---|---|
| `GET /form` | List the form schemas registered on the DMS |
| `GET /entry` | Fetch form entries — the actual experimental records |

Further down, we explore the IMQCAM end point:

| Endpoint | Description |
|---|---|
| `GET /imqcam/count` | Count data files per type across the collection |
| `GET /imqcam/datatype` | List all available data types |
| `GET /imqcam/datafiles` | Query items by data type (pagination, sort, filter, extra fields) |
| `GET /imqcam/partition` | List partitions that have data for a given type |
| `GET /imqcam/partition/details` | Get Dagster partition details for a specific IGSN |

For a field-by-field catalog of every form, see `02_form_catalog.ipynb`.

## Setup

Authentication needs a `GIRDER_API_KEY`. Put it in a `.env` file at the repo root — `get_client()` reads it from there and pins the host to `https://data.imqcam.org/api/v1`.

The `REPO_ROOT` lookup walks up from the working directory until it finds `imqcam.py`, so the notebook works the same whether it is run from the repo root, from `tutorials/`, or from `tutorials/analysis/`.

In [1]:
import json
import sys
from pathlib import Path

from girder_client import HttpError

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imqcam.py").exists())
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client

client = get_client()
print(f"Connected to {client.urlBase}")

Connected to https://data.imqcam.org/api/v1/


## `GET /form`

A form is a schema. Every experimental record on the DMS belongs to exactly one, identified by its `_id` — which is what entries call `formId`.

There are nine.

In [2]:
forms = client.get("form", parameters={"limit": 1000})

print(f"{len(forms)} forms registered\n")
for form in sorted(forms, key=lambda f: f["name"]):
    print(f"  {form['_id']}  {form['name']}")

9 forms registered

  6970da157f6ebb8fb320705c  AM Build Parameters
  68922e35f5b193b7d3e07f5b  Archival ULI Build (simplified)
  68922e94f5b193b7d3e07f5c  Four-point flexural test
  69fa1d0872a32de5fe95028b  Fractography Record
  68ed0eb251d2cf4a1aa1d14f  Micromechanical Simulation Data
  67d39472366ec49ab59dd4db  NASA Ti64 30um Layers Data
  66425a71b18fa1c426e93aa0  Printer Build
  663e6d21b18fa1c426e939ab  Raw Powder Details
  69fa18be514cf501621588cc  Sample Heat Treatment Record


## `GET /entry`

The main data endpoint. Called bare it returns entries from every form at once, but it takes a full set of query parameters — filtering, searching and sorting all happen server-side.

| Parameter | Default | Description |
|---|---|---|
| `formId` | — | Restrict to a single form. Do this rather than fetching everything and filtering in pandas |
| `query` | — | Regex matched against `field` |
| `field` | `sampleId` | Which field `query` searches |
| `limit` | 50 | Page size |
| `offset` | 0 | Pagination offset |
| `sort` | `created` | Field to sort by |
| `sortdir` | 1 | 1 ascending, -1 descending |

In [3]:
# One page, to look at the shape of a single record.
page = client.get("entry", parameters={"limit": 3})
print(f"Returned {len(page)} entries")
print(json.dumps(page[0], indent=2, default=str)[:900])

Returned 3 entries
{
  "_id": "663e8182b18fa1c426e93a7b",
  "created": "2024-05-10T20:20:18.775000+00:00",
  "data": {
    "alloy": "Ti-6Al-4V",
    "batchInformation": [
      {
        "key": "original_power_id",
        "value": "ATI_Ti64_batch_1609_heat_9E64STD004"
      }
    ],
    "characteristics": {
      "maxSize": 53,
      "minSize": 15,
      "tested": true,
      "virginPercent": 0
    },
    "composition": [
      {
        "amount": 0,
        "element": "Ti",
        "isBalance": true,
        "method": "NA (for balance)",
        "unit": "%"
      },
      {
        "amount": 6.09,
        "element": "Al",
        "isBalance": false,
        "method": "WET (Inductively coupled plasma emission)",
        "unit": "%"
      },
      {
        "amount": 3.89,
        "element": "V",
        "isBalance": false,
        "method": "WET (Inductively coupled plasma emission)",
        "unit": "%"



### The entry envelope

Every entry, whatever form it came from, carries the same outer fields. Only `data` differs between forms.

- **`_id`** — the entry's own Girder id
- **`uniqueId`** — the human-readable key the form defines (a build id, a heat-treatment id, a sample IGSN)
- **`formId`** — which schema this entry belongs to
- **`created`, `updated`** — timestamps
- **`folderId`, `folders`, `files`** — attached Girder storage
- **`data`** — the form payload, and the only part whose shape varies

So a generic reader can always page `/entry`, group by `formId`, and treat `data` as the form-specific part.

In [4]:
envelope = {k: v for k, v in page[0].items() if k != "data"}
print("Envelope fields:")
for key, value in sorted(envelope.items()):
    print(f"  {key:12s} {type(value).__name__:6s} {str(value)[:60]}")

print(f"\nPayload (`data`) keys: {sorted(page[0].get('data', {}))}")

Envelope fields:
  _id          str    663e8182b18fa1c426e93a7b
  created      str    2024-05-10T20:20:18.775000+00:00
  files        list   ['663e8179b18fa1c426e93a79']
  folderId     str    663e6e1ab18fa1c426e939ae
  folders      list   []
  formId       str    663e6d21b18fa1c426e939ab
  uniqueId     str    PWD_Ti-6Al-4V_ATI_CMU_001_batch-1609_heat-9E64STD004
  updated      str    2024-05-10T20:20:18.775000+00:00

Payload (`data`) keys: ['alloy', 'batchInformation', 'characteristics', 'composition', 'extraInfo', 'files', 'purchaser', 'rawPowderID', 'sampleNumber', 'supplyCompany']


### Filtering by form

`formId` is applied server-side, so only the form you asked for crosses the wire. Fetching all 758 entries to keep 193 of them is wasted work that gets worse as the collection grows.

In [5]:
FLEXURAL = "68922e94f5b193b7d3e07f5c"

filtered = client.get("entry", parameters={"formId": FLEXURAL, "limit": 1000})
print(f"Server-side formId filter: {len(filtered)} entries")
print(f"All from the requested form: {all(e['formId'] == FLEXURAL for e in filtered)}")

Server-side formId filter: 193 entries
All from the requested form: True


### Three things to watch

- **`limit` is a ceiling, not pagination.** Asking for `limit=1000` returns *at most* 1000 records and gives no indication that it truncated. The collection is under that today, so the bug is latent — but it will start silently dropping data the moment it isn't. Page with `offset` until a short page comes back; `fetch_entries()` in `imqcam.py` does exactly this, and sorts explicitly while doing so, since offset paging over an unordered result set can repeat or skip rows between requests.
- **`data` payloads are not flat.** Several forms nest objects (`buildParameters.infillParameters.laserPower`) and several hold *lists* of repeated measurements (`tests`, `composition`). One entry does not mean one row — see `explode_records()`.
- **The envelope is uniform, the payload is not.** `_id`, `uniqueId`, `formId` and the timestamps are present on every entry whatever its form; everything under `data` varies. Generic readers should rely only on the envelope.

Paging properly gives the true total:

In [9]:
from imqcam import fetch_entries

entries = fetch_entries(client, page_size=100)
print(f"Total entries across all forms: {len(entries)}")

# Compare against a single large request -- equal today, divergent once the
# collection outgrows the ceiling.
single_request = client.get("entry", parameters={"limit": 500})
print(f"Single `limit=500` request:     {len(single_request)}")

# Per-form counts without ever fetching the whole collection.
print()
for form in sorted(forms, key=lambda f: f["name"])[:4]:
    n = len(fetch_entries(client, form_id=form["_id"], page_size=100))
    print(f"  {form['name']:34s} {n:3d}")

Total entries across all forms: 758
Single `limit=500` request:     500

  AM Build Parameters                 27
  Archival ULI Build (simplified)    192
  Four-point flexural test           193
  Fractography Record                 72


## The `/imqcam` endpoints

Everything above reads *entries* — structured records typed into a form. The `/imqcam` endpoints read the other shape: the measurement files themselves, and the derived files that pipelines produce from them.

The unit here is a Girder **item** carrying two pieces of metadata: `data_type` (what kind of measurement) and `igsn` (which specimen). Neither is a form field, so none of `/form`, `/entry` or `formId` applies.

## `GET /imqcam/count`

Counts data files per type across the collection. Optional `baseParentId` and `baseParentType` scope the count to a single Girder collection.

`unclassified` is by far the largest bucket — most files on the DMS have never been assigned a `data_type` and are invisible to every query below.

In [7]:
counts = client.get("imqcam/count")
print(json.dumps(counts, indent=2))

classified = {k: v for k, v in counts.items() if k != "unclassified"}
total = sum(counts.values())
print(f"\n{sum(classified.values()):,} classified files out of {total:,} "
      f"({sum(classified.values()) / total:.1%})")

{
  "ct_tomography": 7148,
  "ebsd_derived": 447,
  "ebsd_raw": 106,
  "fractography_raw": 23,
  "unclassified": 166661
}

7,724 classified files out of 174,385 (4.4%)


### Scoping to a collection

Scoping the count by collection shows which group owns which measurement.

In [8]:
collections = client.get("collection", parameters={"limit": 100})

for collection in collections:
    scoped = client.get("imqcam/count", parameters={
        "baseParentId": collection["_id"],
        "baseParentType": "collection",
    })
    typed = {k: v for k, v in scoped.items() if k != "unclassified"}
    if typed:
        print(f"  {collection['name']:28s} {typed}")

  Characterization             {'fractography_raw': 23}


  Demo                         {'ct_tomography': 1029, 'ebsd_derived': 5, 'ebsd_raw': 1}


  Forms                        {'ebsd_derived': 2}


  IMQCAM Year 2 - Reports and Meetings {'ebsd_derived': 2}


  MS_Reconstruction            {'ebsd_derived': 1, 'ebsd_raw': 12}


...

  NASA                         {'ebsd_raw': 1}


  Processing                   {'ebsd_derived': 158, 'ebsd_raw': 35}


  Rollett                      {'ebsd_derived': 225, 'ebsd_raw': 47}


  Sun Lab Data Sharing         {'ct_tomography': 5118}


  serve_dag_temp               {'ebsd_derived': 8, 'ebsd_raw': 7}


## `GET /imqcam/datatype`

The data types registered on the DMS. No parameters.

The naming carries the raw/derived split directly: `ebsd_raw` is what the microscope wrote, `ebsd_derived` is what the reconstruction pipeline produced from it.

In [9]:
data_types = client.get("imqcam/datatype")
print(json.dumps(data_types, indent=2))

for data_type in data_types:
    if data_type:
        kind = "derived" if data_type.endswith("_derived") else "raw"
        print(f"  {data_type:20s} {kind:8s} {counts.get(data_type, 0):6,d} files")

[
  null,
  "ct_tomography",
  "ebsd_derived",
  "ebsd_raw",
  "fractography_raw"
]
  ct_tomography        raw       7,148 files
  ebsd_derived         derived     447 files
  ebsd_raw             raw         106 files
  fractography_raw     raw          23 files


## `GET /imqcam/datafiles`

The main query endpoint for files. `dataType` is required; everything else is optional.

| Parameter | Default | Description |
|---|---|---|
| `dataType` | — | **Required.** The data type to filter by |
| `limit` | 50 | Page size, **capped at 100** |
| `offset` | 0 | Pagination offset |
| `sort` | `lowerName` | Field to sort by |
| `sortdir` | 1 | 1 ascending, -1 descending |
| `extraFields` | — | JSON array of dotted field paths to include (e.g. `["meta.prov"]`) |
| `filters` | — | JSON object of additional server-side filters |
| `baseParentId` / `baseParentType` | — | Scope to a specific Girder collection |

An item comes back with its Girder envelope — `_id`, `name`, `size`, `folderId`, `created` — plus the `meta` that classified it.

In [10]:
items = client.get("imqcam/datafiles", parameters={"dataType": "ebsd_raw", "limit": 3})

print(f"Fetched {len(items)} items")
print(json.dumps(items[0], indent=2, default=str))

Fetched 3 items
{
  "_id": "699392eaa466c6b4010c3eb3",
  "_modelType": "item",
  "baseParentId": "65b9398c3ef7b1af2165c7c9",
  "baseParentType": "collection",
  "created": "2026-02-16T21:58:02.615000+00:00",
  "creatorId": "67ad6270703d68f56a59a4e7",
  "folderId": "699392d2a466c6b4010c3eae",
  "meta": {
    "data_type": "ebsd_raw",
    "igsn": "CMXMAL00007-036"
  },
  "name": "._CMU_AXFT6_820HIP_300x300ctf.ctf",
  "size": 4096
}


### The count and the query do not agree

`/imqcam/count` counts *files*; `/imqcam/datafiles` returns *items*, and every item it returns carries an `igsn`. Files that were classified but never tied to a specimen are counted and not queryable.

The gap is not small, and for `ct_tomography` it is total: 7,148 files counted, nothing returned. Take the count as an upper bound on what you can retrieve, never as the size of a result set.

In [11]:
def fetch_datafiles(data_type, page_size=100, **params):
    """Page through /imqcam/datafiles until a short page comes back.

    `limit` is capped at 100 server-side and does not signal truncation, so
    paging with `offset` is the only way to get a complete result.
    """
    out, offset = [], 0
    while True:
        batch = client.get("imqcam/datafiles", parameters={
            "dataType": data_type, "limit": page_size, "offset": offset, **params,
        })
        out.extend(batch)
        if len(batch) < page_size:
            return out
        offset += page_size


print(f"{'data type':20s} {'files counted':>14s} {'items retrievable':>18s}")
for data_type in [d for d in data_types if d]:
    retrieved = fetch_datafiles(data_type)
    print(f"{data_type:20s} {counts.get(data_type, 0):14,d} {len(retrieved):18d}")

data type             files counted  items retrievable


ct_tomography                 7,148                  0


ebsd_derived                    447                347


ebsd_raw                        106                 58


fractography_raw                 23                 23


### Sorting

`sort` and `sortdir` order the result server-side. `sortdir=-1` puts the most recent first — the quickest way to see what a pipeline last wrote.

In [12]:
recent = client.get("imqcam/datafiles", parameters={
    "dataType": "ebsd_derived",
    "limit": 5,
    "sort": "created",
    "sortdir": -1,
})

print("5 most recently created ebsd_derived items:")
for item in recent:
    print(f"  {item['created'][:10]}  {item['meta']['igsn']:22s} {item['name']}")

5 most recently created ebsd_derived items:
  2026-06-26  CMXMAL00007-013, CMXMAL00007-031, CMXMAL00007-034, CMXMAL00007-036, CMXMAL00007-040, CMXMAL00007-044 ALPHA_Lath_STATS.xlsx
  2026-06-26  CMXMAL00007-044        AlphaIPFKey.jpg
  2026-06-26  CMXMAL00007-044        AlphaPF.jpg
  2026-06-26  CMXMAL00007-044        AlphaIPF.jpg
  2026-06-26  CMXMAL00007-044        AlphaIPFMAP.jpg


### Extra fields

`extraFields` is a JSON-encoded array of dotted paths, for metadata not returned by default. On a DMS that records provenance this is where `meta.prov` and `meta.runId` come from — which pipeline produced a derived file, and from which input.

IMQCAM does not populate them. Asking returns the same two keys, so a derived file here cannot be traced back to its raw input through metadata alone.

In [13]:
with_prov = client.get("imqcam/datafiles", parameters={
    "dataType": "ebsd_derived",
    "limit": 3,
    "extraFields": '["meta.prov", "meta.runId", "meta.experiment_date"]',
})

for item in with_prov:
    print(f"{item['name'][:40]:42s} {sorted(item.get('meta', {}))}")

1198d4dc-5db7-4d01-a8f7-c029d99a9932.dat   ['data_type', 'igsn']
121924_CMU AlphaGrainSizeMaster.xlsx       ['data_type', 'igsn']
2b9f4dba-c8f3-4fed-b263-945a55fa7947.dat   ['data_type', 'igsn']


### Filtering

`filters` takes a JSON-encoded MongoDB query object, applied server-side on top of `dataType`. Filtering by `meta.igsn` is how you get from a specimen to its files — the join between the two shapes of data.

Note that it must be a JSON *string*, like `extraFields`, not a dict.

In [14]:
igsn = "CMXMAL00011-19"
for data_type in ["ebsd_raw", "ebsd_derived"]:
    matched = fetch_datafiles(data_type, filters=json.dumps({"meta.igsn": igsn}))
    print(f"{igsn}  {data_type:14s} {len(matched):3d} items")

CMXMAL00011-19  ebsd_raw        11 items


CMXMAL00011-19  ebsd_derived    75 items


### IGSNs in file metadata are not always one IGSN

`meta.igsn` is free text, and some derived files were tagged with several specimens at once as a comma-separated string. An exact-match filter will miss those, and grouping on the raw value invents categories that are not samples.

In [15]:
derived = fetch_datafiles("ebsd_derived")
multi = sorted({m for m in (i["meta"]["igsn"] for i in derived) if "," in m})

print(f"{len(multi)} of {len({i['meta']['igsn'] for i in derived})} distinct "
      f"`meta.igsn` values hold more than one IGSN, e.g.")
for value in multi[:3]:
    print(f"  {value!r}")

2 of 11 distinct `meta.igsn` values hold more than one IGSN, e.g.
  'CMXMAL00007-013, CMXMAL00007-031, CMXMAL00007-034, CMXMAL00007-036, CMXMAL00007-040, CMXMAL00007-044'
  'CMXMAL00007-013, CMXMAL00007-031,CMXMAL00007-042'


## `GET /imqcam/partition` and `GET /imqcam/partition/details`

These expose Dagster's partitioning — which `IGSN//date` slices a pipeline has material for, keyed to a checksum so an incremental sync can tell what changed.

**Neither works on this deployment.** `/imqcam/partition` rejects every registered data type with `"Data type <x> is not supported for partitions"`, so `/imqcam/partition/details` has no keys to be called with. The endpoints are part of the shared Girder plugin; the partitioned pipelines behind them are not set up here. They are shown failing rather than left out, so the next person does not spend an afternoon on it.

In [16]:
for data_type in [d for d in data_types if d]:
    try:
        partitions = client.get("imqcam/partition", parameters={"dataType": data_type})
        print(f"  {data_type:20s} OK — {len(partitions)} partitions")
    except HttpError as exc:
        print(f"  {data_type:20s} {exc.status} {json.loads(exc.responseText)['message']}")

  ct_tomography        400 Data type ct_tomography is not supported for partitions.


  ebsd_derived         400 Data type ebsd_derived is not supported for partitions.


  ebsd_raw             400 Data type ebsd_raw is not supported for partitions.


  fractography_raw     400 Data type fractography_raw is not supported for partitions.


`/imqcam/partition/details` takes a `key` from that response. With no partitions to draw from, the shape of the failure is all there is to see — it wants `igsn//experiment_date`, the key format the partition listing would have supplied.

In [17]:
try:
    client.get("imqcam/partition/details", parameters={
        "key": "CMXMAL00011-19", "dataType": "ebsd_derived",
    })
except HttpError as exc:
    print(f"{exc.status} {json.loads(exc.responseText)['message']}")

400 Invalid partition key format. Expected 'igsn//experiment_date',got 'CMXMAL00011-19'.


## Which endpoint to use

| You want | Endpoint | Unit returned |
|---|---|---|
| Process parameters, test results, measured values | `/entry` with `formId` | One entry per record, `data` payload varies by form |
| What forms exist and what they mean | `/form`, then `02_form_catalog.ipynb` | Schemas |
| The measurement files for a specimen | `/imqcam/datafiles` with `filters` on `meta.igsn` | One Girder item per file |
| What file types exist and how much there is | `/imqcam/datatype`, `/imqcam/count` | Type names and file counts |

Both sides page rather than trusting `limit`: `/entry` caps at 1000, `/imqcam/datafiles` at 100, and neither says when it truncated. `fetch_entries()` in `imqcam.py` handles the first; `fetch_datafiles()` above handles the second.

The analysis notebooks in `analysis/` work entirely on the entry side. The file side is where the raw measurements sit, largely unclassified and only partly tied to specimens — worth knowing before planning work that depends on it.